# Erros e aritmética de ponto flutuante

[Abrir este capítulo no Google Colab](https://colab.research.google.com/github/vnikoofard/calculo-numerico-livro/blob/main/livro/02-erros.ipynb)

> **Objetivos.** Ao final deste capítulo você será capaz de:
> 1. explicar como um computador guarda números reais e por que $0{,}1 + 0{,}2 \neq 0{,}3$;
> 2. distinguir erro de **arredondamento** de erro de **truncamento** e prever o passo ótimo quando os dois competem;
> 3. separar um problema **mal condicionado** de um algoritmo **instável**;
> 4. reconhecer e corrigir o cancelamento catastrófico em código, seu ou de um agente.

In [ ]:
# Preparação do ambiente: funciona no seu computador, no Colab e no navegador (JupyterLite).
import sys
try:
    import cnlivro
except ImportError:
    if sys.platform == "emscripten":  # JupyterLite: Python rodando no navegador
        import micropip
        await micropip.install(["plotly", "scipy",
            "https://vnikoofard.github.io/calculo-numerico-livro/pacote/cnlivro-0.1.0-py3-none-any.whl"])
    else:  # Colab ou instalação local sem o pacote
        %pip install -q git+https://github.com/vnikoofard/calculo-numerico-livro

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from cnlivro.estilo import aplicar_estilo, mostrar, grafico_convergencia, grafico_iteracoes, com_controle, numero, CORES
from cnlivro import raizes
from cnlivro.verificacao import ordem_observada, ordem_iterativa, compara

aplicar_estilo()

## 1. Problema de engenharia: o relógio do míssil Patriot

Em 25 de fevereiro de 1991, durante a Guerra do Golfo, uma bateria de mísseis Patriot em Dhahran, na Arábia Saudita, não interceptou um míssil Scud. O Scud atingiu um quartel e matou 28 soldados. A investigação do governo americano (relatório GAO/IMTEC-92-26) encontrou a causa num detalhe aritmético.

O relógio do sistema era um **contador inteiro**: a cada décimo de segundo, ele somava 1. Depois de uma hora, por exemplo, o contador valia 36 000. Essa contagem é exata, porque inteiros são representados sem erro no computador. Para obter o tempo em segundos, o programa multiplicava o contador por $0{,}1$, e o problema estava nesse $0{,}1$: ele **não tem representação exata em binário**. Guardado num registrador de 24 bits, ele vale cerca de $0{,}1 - 9{,}5\times 10^{-8}$. Esse pequeno erro é multiplicado pelo contador, que cresce sem parar. Um erro de cem bilionésimos parece inofensivo. Mas a bateria estava ligada havia **100 horas**:

$$
100 \text{ h} \times 3600 \tfrac{\text{s}}{\text{h}} \times 10 \tfrac{\text{décimos}}{\text{s}} \times 9{,}5\times 10^{-8} \text{ s} \approx 0{,}34 \text{ s}.
$$

Um Scud viaja a cerca de 1 700 m/s. Em 0,34 s, ele percorre **mais de 500 metros**, e o radar procurou o alvo no lugar errado.

Este capítulo explica de onde vêm erros como esse, como eles se propagam e como evitá-los.

In [ ]:
# O erro de representar 0,1 com 23 bits depois da vírgula binária (como no Patriot)
decimo_24bits = int(0.1 * 2**23) / 2**23
erro_por_decimo = 0.1 - decimo_24bits
decimos = 100 * 3600 * 10
print(f"0,1 guardado em 24 bits: {decimo_24bits:.12f}")
print(f"erro por décimo de segundo: {erro_por_decimo:.2e} s")
print(f"erro acumulado em 100 h:    {erro_por_decimo * decimos:.4f} s")
print(f"deslocamento do alvo:       {erro_por_decimo * decimos * 1676:.0f} m  (Scud a 1676 m/s)")

## 2. A ideia em uma imagem

O computador não guarda todos os números reais, só um conjunto finito de **números de máquina**. Entre dois números de máquina vizinhos há um buraco, e todo resultado que cai no buraco é arredondado para um dos lados.

Os números de máquina do `float64` são densos demais para aparecer num gráfico. Por isso a figura usa um sistema de brinquedo, com só 3 bits de mantissa: os números têm a forma $(1{,}b_1b_2b_3)_2 \times 2^e$, com o expoente $e$ indo de $-4$ a $-1$. O painel de cima mostra todos os números positivos desse sistema entre 0 e 0,6. O painel de baixo amplia a região em torno de 0,1.

In [ ]:
def numeros_de_maquina(bits_mantissa, e_min, e_max):
    # Todos os números positivos normalizados (1,b1 b2 ... bp)₂ × 2^e
    valores = []
    for e in range(e_min, e_max + 1):
        for k in range(2**bits_mantissa):
            valores.append((1 + k / 2**bits_mantissa) * 2.0**e)
    return np.array(valores)

nm = numeros_de_maquina(3, -4, -1)
abaixo, acima = nm[nm < 0.1].max(), nm[nm > 0.1].min()   # vizinhos de 0,1 na grade
marca = dict(symbol="line-ns-open", size=22, line=dict(width=2), color=CORES["aproximacao"])

fig = make_subplots(rows=2, cols=1, vertical_spacing=0.3,
                    subplot_titles=("Entre 0 e 0,6: o espaçamento cresce com o valor",
                                    "Ampliação em torno de 0,1: ele cai entre dois vizinhos"))
for linha in (1, 2):
    fig.add_trace(go.Scatter(x=nm, y=np.zeros_like(nm), mode="markers", marker=marca,
                             hovertemplate="%{x}<extra></extra>", name="números de máquina",
                             showlegend=(linha == 1)), row=linha, col=1)
    fig.add_trace(go.Scatter(x=[0.1], y=[0], mode="markers", marker=dict(color=CORES["erro"], size=10),
                             name="0,1 (não é número de máquina)", showlegend=(linha == 1)), row=linha, col=1)

# Painel de cima: o espaçamento dentro de cada intervalo [2^e, 2^(e+1))
for e in range(-4, 0):
    inicio, fim = 2.0**e, min(2.0**(e + 1), 0.6)
    fig.add_annotation(x=(inicio + fim) / 2, y=0.75, text=f"h = 1/{2**(3 - e)}", showarrow=False,
                       font=dict(size=11, color=CORES["referencia"]), row=1, col=1)
    fig.add_vline(x=inicio, line=dict(color=CORES["referencia"], dash="dot", width=1), row=1, col=1)
fig.add_vrect(x0=0.085, x1=0.11, fillcolor=CORES["erro"], opacity=0.12, line_width=0, row=1, col=1)

# Painel de baixo: os dois vizinhos de 0,1 e o arredondamento
for x in (abaixo, acima):
    fig.add_annotation(x=x, y=-0.55, text=str(x).replace(".", ","), showarrow=False,
                       font=dict(size=11, color=CORES["aproximacao"]), row=2, col=1)
fig.add_annotation(x=0.1, y=0, ax=0, ay=-38, text="0,1", showarrow=True, arrowhead=2,
                   arrowcolor=CORES["erro"], font=dict(color=CORES["erro"]), row=2, col=1)

fig.update_xaxes(range=[0, 0.6], dtick=0.1, row=1, col=1)
fig.update_xaxes(range=[0.085, 0.11], dtick=0.005, title="valor", row=2, col=1)
fig.update_yaxes(visible=False, range=[-1, 1.2])
fig.update_layout(height=460, margin=dict(t=50), legend=dict(orientation="h", y=-0.25))
mostrar(fig)
print(f"vizinhos de 0,1: {abaixo} e {acima};  0,1 é arredondado para {min((abaixo, acima), key=lambda x: abs(x - 0.1))}")

**O que notar:** no painel de cima, o espaçamento $h$ entre vizinhos **dobra** a cada potência de 2 (as linhas pontilhadas): $1/128$ entre $1/16$ e $1/8$, $1/64$ entre $1/8$ e $1/4$, e assim por diante. Entre 0 e $1/16$ não há nenhum número, porque o menor expoente do sistema é $-4$. No painel de baixo, $0{,}1$ cai no buraco entre $0{,}09375$ e $0{,}1015625$. Ele é guardado como o vizinho mais próximo, $0{,}1015625$, com erro relativo de cerca de 1,6 %. No `float64` acontece a mesma coisa, só que com 52 bits de mantissa os vizinhos ficam muito mais perto e o erro relativo cai para cerca de $6 \times 10^{-17}$ (seção 3.2).

O **erro relativo** de arredondamento fica sempre abaixo de um mesmo valor, a precisão da máquina, onde quer que o número esteja. É por isso que se fala em "algarismos significativos": o computador guarda um número fixo de dígitos, onde quer que esteja a vírgula.

## 3. Formulação matemática

### 3.1 Erro absoluto e relativo

Se $x$ é o valor verdadeiro e $\tilde x$ a aproximação, há duas maneiras de medir o erro.

O **erro absoluto** $E$ é a diferença entre os dois, na mesma unidade de $x$:

$$
E = x - \tilde x .
$$ (eq-erro-absoluto)

O **erro relativo** $\varepsilon$ divide essa diferença pelo valor verdadeiro. Ele não tem unidade e costuma ser dado em porcentagem:

$$
\varepsilon = \frac{x - \tilde x}{x} = \frac{E}{x} .
$$ (eq-erro-relativo)

O erro relativo é o que importa na maioria das vezes. Errar 1 cm em 1 m ($\varepsilon = 1$ %) é ruim, mas errar 1 cm em 1 km ($\varepsilon = 0{,}001$ %) é excelente. O erro absoluto é o mesmo nos dois casos, e só o relativo mostra a diferença. O erro absoluto é a escolha certa em duas situações: quando o valor verdadeiro é zero ou muito perto de zero, porque aí $\varepsilon$ não está definido ou explode, e quando a especificação é uma tolerância fixa na unidade da grandeza, como "$\pm 0{,}1$ mm" no diâmetro de uma peça usinada. Um resultado com erro relativo $|\varepsilon| \le 0{,}5 \times 10^{-n}$ tem cerca de $n$ **algarismos significativos** corretos.

Nos métodos iterativos não se conhece $x$, e o erro é estimado pela variação entre iterações, $\varepsilon_a = (\tilde x_{\text{novo}} - \tilde x_{\text{velho}})/\tilde x_{\text{novo}}$. É esse o critério de parada dos métodos do capítulo 3.

### 3.2 Ponto flutuante: o padrão IEEE 754

Quase todo computador atual guarda números reais no formato definido pela norma **IEEE 754** (de 1985, revista em 2008 e 2019). O Python, o NumPy e o PyTorch usam esse formato. É ele que explica o erro do Patriot, o `0.1 + 0.2 != 0.3` e o limite de cerca de 16 algarismos significativos. Vale a pena entendê-lo bit a bit.

#### Notação científica, agora em base 2

Na notação científica, escrevemos $-6{,}02 \times 10^{23}$ com três informações: um **sinal** (−), uma **mantissa** (6,02) e um **expoente** (23). A notação é **normalizada** quando há exatamente um algarismo não nulo antes da vírgula. Assim, cada número tem uma única representação: $602 \times 10^{21}$ e $0{,}602 \times 10^{24}$ não valem.

Em base 2 é igual, com uma vantagem. O único algarismo não nulo em binário é o 1, então todo número normalizado começa com "1,". Por exemplo:

$$
5{,}75 = 4 + 1 + \tfrac12 + \tfrac14 = (101{,}11)_2 = (1{,}0111)_2 \times 2^{2}.
$$

Deslocar a vírgula binária uma casa para a esquerda divide o número por 2, e por isso o expoente sobe uma unidade a cada casa. Como o primeiro bit da mantissa é **sempre** 1, ele não precisa ser guardado. É o **bit implícito**: um bit de precisão de graça.

#### Os três campos de um `float64`

Um número de precisão dupla (`float64`, o padrão do Python e do NumPy) ocupa 64 bits, divididos em três campos:

| campo | bits | o que guarda |
| --- | --- | --- |
| sinal $s$ | 1 | 0 para positivo, 1 para negativo |
| expoente $e$ | 11 | o expoente verdadeiro **somado a 1023** (o viés) |
| fração $f$ | 52 | os bits da mantissa depois da vírgula, $f = f_1 f_2 \dots f_{52}$ |

O valor representado é

$$
x = (-1)^{s} \times (1{,}f_1 f_2 \dots f_{52})_2 \times 2^{\,e - 1023}, \qquad 1 \le e \le 2046 .
$$ (eq-ieee754)

**Por que um viés?** O expoente verdadeiro $E$ pode ser negativo: $0{,}1 \approx 1{,}6 \times 2^{-4}$. Em vez de gastar um bit com o sinal do expoente, a norma guarda $e = E + 1023$, que é sempre um inteiro sem sinal. Com 11 bits, $e$ vai de 0 a 2047. Os valores $e = 0$ (todos os bits 0) e $e = 2047$ (todos os bits 1) ficam reservados para casos especiais, vistos adiante. Sobram $1 \le e \le 2046$, ou seja, $-1022 \le E \le 1023$. O viés tem um bônus: para números positivos, comparar os 64 bits como se fossem inteiros dá a mesma ordem que comparar os números, e o hardware compara floats tão rápido quanto inteiros.

**Mantissa e fração.** A mantissa é

$$
(1{,}f_1 f_2 \dots f_{52})_2 = 1 + \frac{f_1}{2} + \frac{f_2}{4} + \dots + \frac{f_{52}}{2^{52}},
$$

um número sempre em $[1, 2)$. São 52 bits guardados mais o implícito, ou seja, **53 bits de precisão**. Isso equivale a $53 \log_{10} 2 \approx 16$ algarismos decimais.

A figura mostra os 64 bits do número $0{,}1$, com cada campo numa cor. Bits iguais a 1 têm cor escura; bits iguais a 0, cor clara.

In [ ]:
import struct
from decimal import Decimal, getcontext
getcontext().prec = 60

def campos(x):
    """Os três campos de um float64: sinal, expoente e fração, como textos de bits."""
    inteiro = struct.unpack(">Q", struct.pack(">d", x))[0]   # os mesmos 64 bits, lidos como inteiro
    bits = f"{inteiro:064b}"
    return bits[0], bits[1:12], bits[12:]

def _decimal(d, algarismos=22):
    """Decimal exato, encurtado para leitura, com vírgula decimal."""
    texto = f"{d:.{algarismos}g}" if abs(d) >= Decimal("1e-6") and abs(d) < Decimal("1e7") else f"{d:.{algarismos - 6}e}"
    if Decimal(texto) != d:
        texto += "…"
    return texto.replace(".", ",")

def decodifica(x):
    """Linhas de texto que explicam, campo a campo, o valor guardado."""
    s, e_bits, f_bits = campos(x)
    e, f = int(e_bits, 2), int(f_bits, 2)
    sinal = "−" if s == "1" else "+"
    linha_s = f"sinal: s = {s}  →  {sinal}"
    if e == 2047:
        return [linha_s, "expoente: e = 2047 (todos os bits 1)  →  reservado",
                "fração: f = 0  →  infinito" if f == 0 else "fração: f ≠ 0  →  NaN (não é um número)",
                f"valor: {sinal}∞" if f == 0 else "valor: NaN"]
    if e == 0:
        mant = Decimal(f) / Decimal(2**52)
        linha_e = "expoente: e = 0 (todos os bits 0)  →  reservado: zero ou subnormal, E = −1022"
        linha_m = f"mantissa SEM o bit implícito: (0,f)₂ = {_decimal(mant)}"
        valor = mant * Decimal(2) ** -1022
    else:
        mant = 1 + Decimal(f) / Decimal(2**52)
        linha_e = f"expoente: e = {e}  →  E = e − 1023 = {e - 1023}".replace("-", "−")
        linha_m = f"mantissa: (1,f)₂ = {_decimal(mant)}"
        valor = mant * Decimal(2) ** (e - 1023)
    valor = -valor if s == "1" else valor
    return [linha_s, linha_e, linha_m, f"valor guardado: {_decimal(valor, 28)}"]

def figura_bits(x, titulo=None):
    s, e_bits, f_bits = campos(x)
    bits = s + e_bits + f_bits
    campo = [0] + [1] * 11 + [2] * 52
    z = [[2 * c + int(b) for c, b in zip(campo, bits)]]
    claro_escuro = ["#F6D3BE", "#D55E00", "#C7DDF0", "#0072B2", "#C3E8DC", "#009E73"]
    escala = [[k / 6 + d, cor] for k, cor in enumerate(claro_escuro) for d in (0, 1 / 6)]
    fig = go.Figure(go.Heatmap(z=z, x=list(range(64)), y=[0], zmin=-0.5, zmax=5.5, colorscale=escala,
                               showscale=False, xgap=1, hoverinfo="skip"))
    for valor_bit, cor in (("0", "#333333"), ("1", "white")):
        pos = [i for i, b in enumerate(bits) if b == valor_bit]
        fig.add_trace(go.Scatter(x=pos, y=[0] * len(pos), mode="text", text=[valor_bit] * len(pos),
                                 textfont=dict(size=11, color=cor), hoverinfo="skip", showlegend=False))
    linhas = decodifica(x)
    fig.add_trace(go.Scatter(x=[-0.5] * len(linhas), y=[-1.1 - 0.55 * k for k in range(len(linhas))],
                             mode="text", text=linhas, textposition="middle right",
                             textfont=dict(size=14), hoverinfo="skip", showlegend=False))
    for x_rot, texto, cor in [(-0.4, "s", "#D55E00"), (0.6, "expoente e (11 bits)", "#0072B2"),
                              (30, "fração f (52 bits)", "#009E73")]:
        fig.add_annotation(x=x_rot, y=0.85, text=texto, showarrow=False, xanchor="left",
                           font=dict(size=13, color=cor))
    fig.update_xaxes(visible=False, range=[-1, 64])
    fig.update_yaxes(visible=False, range=[-3.4, 1.1])
    fig.update_layout(width=560, height=300, margin=dict(l=10, r=10, t=50, b=10), showlegend=False,
                      title=titulo or f"Os 64 bits de {numero(x)} em float64")
    return fig

mostrar(figura_bits(0.1, "Os 64 bits de 0,1 em float64"))

**O que notar:** o expoente guardado é $e = 1019$, e não $-4$, por causa do viés. A fração é o padrão $1001$ repetido, cortado em 52 bits, e o valor guardado é um pouco maior que 0,1. O exemplo 3 abaixo explica por quê.

#### Decodificando passo a passo

Para codificar um número: (1) escreva-o em binário, (2) normalize para $(1{,}\dots)_2 \times 2^E$, (3) guarde $e = E + 1023$ e os bits depois da vírgula. Para decodificar, faça o caminho inverso com a equação {eq}`eq-ieee754`.

**Exemplo 1: $x = 1$.** Já está normalizado: $1 = (1{,}0)_2 \times 2^0$.

- sinal: $s = 0$;
- expoente: $E = 0$, então $e = 0 + 1023 = 1023 = (01111111111)_2$;
- fração: $f = 000\dots0$ (52 zeros).

Em hexadecimal, que agrupa os bits de 4 em 4, é `3FF0000000000000`.

**Exemplo 2: $x = -2{,}5$.** Em binário, $2{,}5 = 2 + \tfrac12 = (10{,}1)_2 = (1{,}01)_2 \times 2^1$.

- sinal: $s = 1$, porque o número é negativo;
- expoente: $E = 1$, então $e = 1024 = (10000000000)_2$;
- fração: os bits depois da vírgula de $(1{,}01)_2$, completados com zeros: $f = 01000\dots0$.

Em hexadecimal: `C004000000000000`. Decodificando de volta: $(-1)^1 \times (1 + \tfrac04 + \tfrac14) \times 2^{1024 - 1023} = -1{,}25 \times 2 = -2{,}5$.

**Exemplo 3: $x = 0{,}1$.** A seção 4 mostra, à mão, que $0{,}1 = (0{,}0\overline{0011})_2$, uma dízima periódica. Normalizando (a vírgula anda 4 casas para a direita):

$$
0{,}1 = (1{,}\overline{1001})_2 \times 2^{-4}.
$$

- sinal: $s = 0$;
- expoente: $E = -4$, então $e = -4 + 1023 = 1019 = (01111111011)_2$;
- fração: $1001\,1001\,1001\,\dots$, que **não cabe** em 52 bits. Os 52 primeiros bits são treze blocos $1001$. Os bits seguintes, $1001\dots$, valem mais que meia unidade da última casa, então a norma **arredonda para cima**: o último bloco vira $1010$.

O valor guardado é

$$
\mathrm{fl}(0{,}1) = 0{,}1000000000000000055511151231257827\ldots,
$$

um pouco **maior** que 0,1, com erro relativo de $5{,}6 \times 10^{-17}$. Em hexadecimal: `3FB999999999999A`, com o `9` repetido (que é $1001$) e o `A` final (que é $1010$).

Esses três exemplos são um bom teste para qualquer código que manipule bits de floats, seja o seu, seja o de um agente:

In [ ]:
import struct

def campos(x):
    """Os três campos de um float64: sinal, expoente e fração, como textos de bits."""
    inteiro = struct.unpack(">Q", struct.pack(">d", x))[0]   # os mesmos 64 bits, lidos como inteiro
    bits = f"{inteiro:064b}"
    return bits[0], bits[1:12], bits[12:]
for x in [1.0, -2.5, 0.1]:
    s, e, f = campos(x)
    print(f"{numero(x):>5}:  s = {s}   e = {e} ({int(e, 2)})   f = {f[:12]}…   hex = {struct.pack('>d', x).hex().upper()}")

#### O maior, o menor e os casos especiais

O **maior** `float64` tem o maior expoente permitido, $e = 2046$, e todos os bits da fração iguais a 1:

$$
x_{\max} = (1{,}11\dots1)_2 \times 2^{1023} = (2 - 2^{-52}) \times 2^{1023} \approx 1{,}80 \times 10^{308}.
$$

O **menor positivo normalizado** tem $e = 1$ e fração nula: $x_{\min} = 2^{-1022} \approx 2{,}23 \times 10^{-308}$.

Os dois valores reservados do expoente codificam os casos especiais:

| expoente $e$ | fração $f$ | significado | exemplo |
| --- | --- | --- | --- |
| 0 | 0 | $\pm 0$ (há dois zeros, com sinais diferentes) | `-0.0 == 0.0` é `True` |
| 0 | $\neq 0$ | **subnormal**: $(-1)^s \times (0{,}f)_2 \times 2^{-1022}$ | o menor positivo é $2^{-1074} \approx 4{,}9 \times 10^{-324}$ |
| 2047 | 0 | $\pm\infty$ | `1e308 * 10` |
| 2047 | $\neq 0$ | NaN (*not a number*) | `inf - inf`, ou `0/0` no NumPy |

**Subnormais.** Sem eles, haveria um buraco entre 0 e $2^{-1022}$, como o buraco entre 0 e $1/16$ da figura da seção 2. Os subnormais abrem mão do bit implícito, usam $(0{,}f)_2$ e preenchem esse buraco com pontos igualmente espaçados de $2^{-1074}$. É o **underflow gradual**: com ele, $x - y = 0$ só quando $x = y$. O preço é a precisão, porque um subnormal tem menos bits significativos (o menor deles tem um só).

**Infinito e NaN.** Em vez de parar o programa, uma conta que transborda devolve $\pm\infty$, e uma conta sem sentido devolve NaN. A conta segue, e o problema aparece no fim. Um NaN contamina tudo o que toca: $\text{NaN} + 1 = \text{NaN}$. Além disso, NaN é o único valor diferente de si mesmo (`nan == nan` é `False`), por isso o teste certo é `np.isnan(x)`. O Python puro é mais rígido que o NumPy: `0.0 / 0.0` levanta `ZeroDivisionError` em vez de devolver NaN.

#### A grade de números e a precisão da máquina

Fixado o expoente $E$, os números de máquina entre $2^E$ e $2^{E+1}$ são

$$
\left(1 + \frac{k}{2^{52}}\right) \times 2^{E}, \qquad k = 0, 1, \dots, 2^{52} - 1 .
$$

São $2^{52} \approx 4{,}5 \times 10^{15}$ números **igualmente espaçados**, a uma distância de $2^{E - 52}$ um do outro. Ao passar para o intervalo seguinte, o espaçamento dobra. É exatamente a figura da seção 2, com 52 bits de fração em vez de 3.

A grandeza central é a **precisão da máquina**, a distância entre 1 e o próximo número de máquina:

$$
\varepsilon_{\text{mach}} = 2^{-52} \approx 2{,}22 \times 10^{-16}.
$$

Como o espaçamento absoluto cresce junto com o número, o espaçamento **relativo** fica sempre entre $\varepsilon_{\text{mach}}/2$ e $\varepsilon_{\text{mach}}$. A figura mostra os dois, para `float32` e `float64`.

In [ ]:
x = np.logspace(-3, 3, 3000)
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.14,
                    subplot_titles=("Espaçamento absoluto: cresce com x", "Espaçamento relativo, em unidades de ε"))
for tipo, cor, nome in [(np.float32, CORES["secundaria"], "float32"), (np.float64, CORES["aproximacao"], "float64")]:
    xt = x.astype(tipo)
    passo = np.spacing(xt).astype(float)          # distância até o próximo número de máquina
    eps = float(np.finfo(tipo).eps)
    fig.add_trace(go.Scatter(x=x, y=passo, name=nome, line=dict(color=cor, shape="hv")), row=1, col=1)
    fig.add_trace(go.Scatter(x=x, y=passo / xt.astype(float) / eps, name=nome, showlegend=False,
                             line=dict(color=cor, dash="dot" if nome == "float64" else "solid")), row=1, col=2)
for nivel, texto in [(1, "ε"), (0.5, "ε/2")]:
    fig.add_hline(y=nivel, line=dict(color=CORES["referencia"], dash="dash", width=1), row=1, col=2,
                  annotation_text=texto, annotation_position="right")
fig.update_xaxes(type="log", title="x", exponentformat="power", dtick=1)
fig.update_yaxes(type="log", exponentformat="power", title="distância até o próximo float", row=1, col=1)
fig.update_yaxes(range=[0.4, 1.1], title="distância relativa / ε", row=1, col=2)
fig.update_layout(width=640, height=380, legend=dict(orientation="h", y=-0.25), margin=dict(r=40))
mostrar(fig)
print(f"ε float64 = {np.finfo(np.float64).eps:.3e} = 2^-52;   ε float32 = {np.finfo(np.float32).eps:.3e} = 2^-23")

**O que notar:** à esquerda, o espaçamento sobe em degraus, dobrando a cada potência de 2, e o `float32` fica cerca de $2^{29} \approx 5 \times 10^8$ vezes acima do `float64`. À direita, o espaçamento relativo, dividido pelo $\varepsilon$ de cada formato, faz um "dente de serra" que nunca sai da faixa entre $\varepsilon/2$ e $\varepsilon$. As duas curvas coincidem: o desenho é o mesmo nos dois formatos, só a escala $\varepsilon$ muda. É por isso que o erro de arredondamento se mede em termos **relativos**.

#### Arredondamento ao mais próximo

Quando o resultado exato de uma operação não é um número de máquina, a norma o **arredonda para o número de máquina mais próximo**. Em caso de empate, quando o resultado cai exatamente no meio de dois vizinhos, ela escolhe o vizinho cujo último bit é 0 (**empate para o par**). Desempatar sempre para cima criaria um viés que se acumula em somas longas, e desempatar para o par sobe na metade das vezes e desce na outra metade.

A distância até o vizinho mais próximo é no máximo metade do espaçamento. Daí vem o modelo de todo o estudo de erros de arredondamento: cada operação aritmética devolve o resultado exato com um erro relativo de no máximo $u = \varepsilon_{\text{mach}}/2$, a **unidade de arredondamento**:

$$
\mathrm{fl}(a \circ b) = (a \circ b)(1 + \delta), \qquad |\delta| \le u = \tfrac12\varepsilon_{\text{mach}} .
$$ (eq-modelo-fl)

Cada operação introduz um erro relativo de no máximo $1{,}1\times 10^{-16}$, cerca de 16 algarismos significativos. O problema não é uma operação isolada: é o que acontece quando os erros se **acumulam** (como no Patriot) ou são **amplificados**.

**Por que `0.1 + 0.2 != 0.3`.** Agora dá para seguir a conta bit a bit. Os valores guardados de 0,1 e 0,2 são ambos um pouco maiores que os verdadeiros (exemplo 3). A soma exata desses dois valores cai **exatamente no meio** entre dois floats vizinhos: o número de máquina mais próximo de 0,3 e o seguinte. É um empate, e o empate vai para o vizinho de último bit par, que é o de cima. O resultado fica um espaçamento acima de $\mathrm{fl}(0{,}3)$, a uma distância de $2^{-54} \approx 5{,}6 \times 10^{-17}$:

In [ ]:
from fractions import Fraction
soma_exata = Fraction(0.1) + Fraction(0.2)        # soma exata dos valores GUARDADOS de 0,1 e 0,2
vizinho_baixo, vizinho_cima = Fraction(0.3), Fraction(np.nextafter(0.3, 1.0))
print("a soma exata está no meio dos vizinhos?", soma_exata - vizinho_baixo == vizinho_cima - soma_exata)
print(f"0.1 + 0.2 = {0.1 + 0.2!r}   hex {(0.1 + 0.2).hex()}")
print(f"0.3       = {0.3!r}   hex {(0.3).hex()}   (último dígito hexadecimal 3 é ímpar, 4 é par)")
print("1 + 2^-53 == 1 ?", 1 + 2**-53 == 1, "  (empate entre 1 e 1 + ε: vai para o par, que é 1)")
print("1 + 2^-52 == 1 ?", 1 + 2**-52 == 1)

#### Outros formatos: `float32`, `float16` e `bfloat16`

A norma define formatos de vários tamanhos, todos com a mesma ideia: sinal, expoente com viés e fração com bit implícito. Muda só a quantidade de bits em cada campo.

| formato | bits (sinal / expoente / fração) | viés | $\varepsilon_{\text{mach}}$ | maior valor | menor normal | algarismos |
| --- | --- | --- | --- | --- | --- | --- |
| `float16` | 1 / 5 / 10 | 15 | $9{,}8 \times 10^{-4}$ | 65 504 | $6{,}1 \times 10^{-5}$ | ≈ 3 |
| `bfloat16` | 1 / 8 / 7 | 127 | $7{,}8 \times 10^{-3}$ | $3{,}4 \times 10^{38}$ | $1{,}2 \times 10^{-38}$ | ≈ 2 |
| `float32` | 1 / 8 / 23 | 127 | $1{,}2 \times 10^{-7}$ | $3{,}4 \times 10^{38}$ | $1{,}2 \times 10^{-38}$ | ≈ 7 |
| `float64` | 1 / 11 / 52 | 1023 | $2{,}2 \times 10^{-16}$ | $1{,}8 \times 10^{308}$ | $2{,}2 \times 10^{-308}$ | ≈ 16 |

Os bits do **expoente** decidem a **faixa** de valores, e os bits da **fração** decidem a **precisão**. O `bfloat16`, criado para aprendizado de máquina, mantém os 8 bits de expoente do `float32` (a mesma faixa) e corta a fração para 7 bits. Em redes neurais, transbordar é pior que perder precisão. Já o `float16` tem mais precisão, mas transborda acima de 65 504.

**Atenção ao PyTorch.** O NumPy usa `float64` por padrão, mas o PyTorch usa **`float32`**: `torch.tensor(0.1)` guarda $0{,}1$ com só 24 bits de precisão, como o Patriot. Quando um agente traduz código de NumPy para PyTorch, os resultados podem mudar na oitava casa decimal. Isso não é erro do agente: é o formato. A função `torch.finfo` dá a tabela acima para os tipos do PyTorch, e `np.finfo`, para os do NumPy:

In [ ]:
print(f"{'tipo':>8} {'fração':>7} {'ε':>9} {'maior':>9} {'menor normal':>13} {'menor subnormal':>16}")
for tipo in [np.float16, np.float32, np.float64]:
    i = np.finfo(tipo)
    print(f"{tipo.__name__:>8} {i.nmant:7d} {float(i.eps):9.2e} {float(i.max):9.2e} "
          f"{float(i.tiny):13.2e} {float(i.smallest_subnormal):16.1e}")

### 3.3 Cancelamento catastrófico

Subtrair dois números quase iguais **amplifica** o erro relativo. Suponha que $a$ e $b$ já cheguem à subtração com pequenos erros relativos $\delta_a$ e $\delta_b$, ambos de tamanho no máximo $\delta$ (por exemplo, $\delta = u$ quando vêm de um arredondamento). Os valores guardados são $\tilde a = a(1 + \delta_a)$ e $\tilde b = b(1 + \delta_b)$. Mesmo que a subtração em si seja exata, o **erro absoluto** da diferença, como na equação {eq}`eq-erro-absoluto`, é

$$
E_{a-b} = (a - b) - (\tilde a - \tilde b) = -(a\,\delta_a - b\,\delta_b), \qquad |E_{a-b}| \le \delta\,(|a| + |b|).
$$

Dividindo por $|a - b|$, obtemos o **erro relativo** da diferença, como na equação {eq}`eq-erro-relativo`:

$$
|\varepsilon_{a-b}| = \frac{|E_{a-b}|}{|a - b|} \le \delta\,\frac{|a| + |b|}{|a - b|}.
$$

O fator $(|a| + |b|)/|a - b|$ é o **amplificador**. Quando $a$ e $b$ são parecidos, o numerador é grande e o denominador é pequeno. Se $a$ e $b$ concordam nos primeiros 10 dígitos, o fator é da ordem de $10^{10}$: um erro relativo de $10^{-16}$ nas entradas vira $10^{-6}$ na diferença, e dos 16 algarismos sobram 6. O exemplo clássico é a fórmula de Bhaskara quando $b^2 \gg 4ac$:

$$
x_1 = \frac{-b + \sqrt{b^2 - 4ac}}{2a}.
$$

Para $b > 0$, $\sqrt{b^2 - 4ac} \approx b$ e a soma no numerador cancela. A saída é reescrever a expressão. Multiplicando em cima e embaixo pelo conjugado:

$$
x_1 = \frac{-2c}{b + \sqrt{b^2 - 4ac}},
$$

que só **soma** números de mesmo sinal. A matemática é a mesma; o comportamento numérico, não.

### 3.4 Erro de truncamento e a série de Taylor

O **erro de truncamento** vem de trocar um processo infinito por um finito. A ferramenta para medi-lo é a série de Taylor com resto:

$$
f(x + h) = f(x) + f'(x)\,h + \frac{f''(x)}{2!}h^2 + \dots + \frac{f^{(n)}(x)}{n!}h^n + R_n, \qquad R_n = \frac{f^{(n+1)}(\xi)}{(n+1)!}h^{n+1}.
$$

Por exemplo, isolando $f'(x)$ com $n = 1$:

$$
f'(x) = \frac{f(x+h) - f(x)}{h} - \frac{f''(\xi)}{2}\, h .
$$

A **diferença progressiva** tem erro de truncamento $O(h)$: diminuir $h$ diminui o erro.

### 3.5 Quando os dois erros competem

Diminuir $h$ reduz o truncamento, mas a subtração $f(x+h) - f(x)$ sofre cancelamento quando $h$ é pequeno. Se cada valor de $f$ tem erro de arredondamento da ordem de $\varepsilon_{\text{mach}}|f|$, o erro total é

$$
E(h) \approx \underbrace{\frac{|f''|}{2}\,h}_{\text{truncamento}} + \underbrace{\frac{2\,\varepsilon_{\text{mach}}|f|}{h}}_{\text{arredondamento}} .
$$

O mínimo fica em $h^* \approx 2\sqrt{\varepsilon_{\text{mach}}|f/f''|} \sim 10^{-8}$, e o melhor erro possível é da ordem de $\sqrt{\varepsilon_{\text{mach}}} \approx 10^{-8}$. **Diminuir $h$ além de $h^*$ piora o resultado.** A seção 5 mostra isso num gráfico.

### 3.6 Condicionamento e estabilidade

Quando um resultado numérico sai errado, há dois culpados possíveis: o **problema** e o **algoritmo**. Vale a pena separá-los.

#### Condicionamento: o problema é sensível?

Antes de pensar em algoritmo, perguntamos se o próprio problema é sensível. Se a entrada $x$ já chega com um pequeno erro, vindo de uma medida ou de um arredondamento, quanto isso estraga a resposta $f(x)$, **mesmo que o cálculo seja exato**?

**De onde sai a fórmula.** Suponha que, no lugar de $x$, a entrada seja $x(1 + \delta)$, isto é, $x$ com erro relativo $\delta$. Pela série de Taylor de primeira ordem,

$$
f(x + x\delta) \approx f(x) + f'(x)\, x\,\delta .
$$

O erro relativo na saída é então

$$
\frac{f(x + x\delta) - f(x)}{f(x)} \approx \underbrace{\frac{x\, f'(x)}{f(x)}}_{\text{fator de amplificação}}\;\delta .
$$

O valor absoluto desse fator é o **número de condição**:

$$
\kappa(x) = \left|\frac{x\, f'(x)}{f(x)}\right| = \frac{\text{erro relativo que sai}}{\text{erro relativo que entra}} .
$$ (eq-numero-condicao)

Como $\kappa$ é uma razão entre erros **relativos**, aparecem o $x$ no numerador e o $f(x)$ no denominador: $f'(x)$ sozinho mediria a sensibilidade em erros absolutos. Um erro relativo $\delta$ na entrada vira cerca de $\kappa\,\delta$ na saída. Se $\kappa$ é pequeno (da ordem de 1), o problema é **bem condicionado**. Se $\kappa$ é grande, ele é **mal condicionado**.

**Exemplos.**

- $f(x) = \sqrt{x}$: $f'(x) = 1/(2\sqrt x)$, então $\kappa = \left|x \cdot \frac{1}{2\sqrt x} \cdot \frac{1}{\sqrt x}\right| = \tfrac12$ para todo $x > 0$. O problema é bem condicionado: o erro relativo até **diminui** pela metade.
- $f(x) = x - 1$ perto de $x = 1$: $\kappa = |x/(x - 1)|$. Em $x = 1{,}0001$, $\kappa \approx 10^4$, e um erro de $10^{-16}$ na entrada vira $10^{-12}$ na saída. É o cancelamento catastrófico da seção 3.3, visto agora como condicionamento: o amplificador $(|a| + |b|)/|a - b|$ daquela seção é o número de condição da subtração.
- $f(x) = \tan x$ perto de $\pi/2$: $\kappa = |x/(\sin x \cos x)|$, que cresce sem limite quando $\cos x \to 0$. A seção 7.4 mostra isso em números.

**Por que nenhum algoritmo resolve.** O número de condição depende só de $f$ e de $x$, e não de **como** se calcula $f$. Se a entrada já tem erro $\delta$, nem um computador com precisão infinita evita o erro $\kappa\,\delta$ na saída. O mau condicionamento só se combate mudando o **problema**: medindo a entrada com mais precisão ou reformulando a pergunta.

#### Estabilidade: o algoritmo acrescenta erro?

A **estabilidade** é uma propriedade do **algoritmo**: ele introduz erros muito maiores que os inevitáveis, isto é, maiores que $\kappa$ vezes o erro de arredondamento das entradas? A fórmula de Bhaskara para $x_1$ é **instável** quando $b^2 \gg 4ac$, embora o problema (achar a raiz) seja bem condicionado. A fórmula alternativa, que resolve o mesmo problema, é estável.

Resumindo, o erro final de um cálculo tem duas parcelas:

$$
\text{erro final} \;\approx\; \underbrace{\kappa \times \text{erro da entrada}}_{\text{do problema}} \;+\; \underbrace{\text{erro introduzido pelo algoritmo}}_{\text{da implementação}} .
$$

A primeira parcela não depende de quem programou. A segunda depende, e é nela que um código, seu ou de um agente, pode ser melhorado.

### 3.7 Propagação de erros

Se $y = f(x_1, \dots, x_n)$ e cada $x_i$ tem incerteza $\Delta x_i$, a aproximação de primeira ordem por Taylor dá

$$
\Delta y \approx \sum_{i=1}^{n} \left|\frac{\partial f}{\partial x_i}\right| \Delta x_i .
$$

Essa fórmula é a base da análise de incertezas em medidas de engenharia. O apêndice A a estende com Monte Carlo e diferenciação automática.

## 4. À mão

**0,1 em binário.** Multiplicando a parte fracionária por 2 repetidamente e anotando a parte inteira:

| passo | valor × 2 | dígito binário |
| --- | --- | --- |
| 1 | 0,2 | 0 |
| 2 | 0,4 | 0 |
| 3 | 0,8 | 0 |
| 4 | 1,6 | 1 |
| 5 | 1,2 | 1 |
| 6 | 0,4 | 0 |
| 7 | 0,8 | 0 |
| 8 | 1,6 | 1 |

A partir do passo 6, o ciclo 0, 0, 1, 1 se repete: $0{,}1 = (0{,}0\overline{0011})_2$. Como a dízima é infinita, qualquer número finito de bits a corta, e daí vem o erro do Patriot.

**Bhaskara com 4 algarismos.** Para $x^2 + 62{,}10\,x + 1 = 0$, a raiz menor é $x_1 \approx -0{,}01610$. Com 4 algarismos significativos, $\sqrt{62{,}10^2 - 4} = \sqrt{3852} = 62{,}06$, e

$$
x_1 = \frac{-62{,}10 + 62{,}06}{2} = -0{,}02000 \quad (\text{erro de } 24\,\%), \qquad
x_1 = \frac{-2}{62{,}10 + 62{,}06} = -0{,}01611 \quad (\text{erro de } 0{,}06\,\%).
$$

In [ ]:
print(f"0,1 + 0,2 = {0.1 + 0.2!r}")
print(f"0,1 em binário (como guardado): {0.1:.55f}")
print(f"precisão da máquina: {np.finfo(float).eps:.3e}  =  2^-52 = {2.0**-52:.3e}")

## 5. Explore

### 5.1 Taylor: quantos termos são suficientes?

O slider muda o número de termos da série de Taylor de $\cos x$ em torno de $x_0 = 0$. Perto de $x_0$ poucos termos bastam; longe dele, são necessários muitos.

In [ ]:
from math import factorial

def taylor_cos(x, n):
    return sum((-1) ** k * x ** (2 * k) / factorial(2 * k) for k in range(n + 1))

def figura_taylor(n):
    x = np.linspace(-2 * np.pi, 2 * np.pi, 400)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=np.cos(x), name="cos x", line=dict(color=CORES["exata"])))
    fig.add_trace(go.Scatter(x=x, y=taylor_cos(x, n), name=f"Taylor, grau {2 * n}", line=dict(color=CORES["aproximacao"])))
    fig.update_xaxes(title="x (rad)")
    fig.update_yaxes(title="y", range=[-2, 2])
    fig.update_layout(title=f"Série de Taylor de cos x até o grau {2 * n}")
    return fig

quadros = [(str(2 * n), figura_taylor(n)) for n in range(0, 10)]
mostrar(com_controle(quadros, rotulo="grau = ", ativo=2, titulos=[q[1].layout.title.text for q in quadros]))

### 5.2 O erro total em forma de V

Erro da derivada numérica de $\sin x$ em $x = 1$, para $h$ de $10^{-1}$ até $10^{-17}$. Os pontos são o erro medido; as linhas tracejadas, as estimativas da seção 3.5.

In [ ]:
h = np.logspace(-1, -17, 65)
x0 = 1.0
erro_prog = np.abs((np.sin(x0 + h) - np.sin(x0)) / h - np.cos(x0))
erro_cent = np.abs((np.sin(x0 + h) - np.sin(x0 - h)) / (2 * h) - np.cos(x0))
eps = np.finfo(float).eps

fig = go.Figure()
fig.add_trace(go.Scatter(x=h, y=erro_prog, mode="markers", name="diferença progressiva", marker=dict(color=CORES["erro"])))
fig.add_trace(go.Scatter(x=h, y=erro_cent, mode="markers", name="diferença central", marker=dict(color=CORES["secundaria"])))
fig.add_trace(go.Scatter(x=h, y=np.sin(x0) / 2 * h, mode="lines", name="truncamento ≈ |f''| h / 2",
                         line=dict(color=CORES["referencia"], dash="dash")))
fig.add_trace(go.Scatter(x=h, y=2 * eps * np.sin(x0) / h, mode="lines", name="arredondamento ≈ 2 ε |f| / h",
                         line=dict(color=CORES["referencia"], dash="dot")))
fig.update_xaxes(type="log", title="passo h", exponentformat="power", autorange="reversed")
fig.update_yaxes(type="log", title="erro absoluto", range=[-12, 1], exponentformat="power")
fig.update_layout(title="Diminuir h ajuda até certo ponto; depois atrapalha")
mostrar(fig)
i = np.argmin(erro_prog)
print(f"melhor h (progressiva): {h[i]:.1e}, erro {erro_prog[i]:.1e}")
j = np.argmin(erro_cent)
print(f"melhor h (central):     {h[j]:.1e}, erro {erro_cent[j]:.1e}")

**O que notar:** o eixo horizontal vai de $h$ grande, à esquerda, para $h$ pequeno, à direita. À esquerda domina o truncamento: diminuindo $h$, o erro cai. À direita domina o arredondamento, e o erro **sobe** com $1/h$. O melhor $h$ fica perto de $10^{-8}$ para a diferença progressiva. A diferença central, de ordem 2, atinge um erro menor, com um $h$ maior, perto de $10^{-5}$. No extremo direito, $h < \varepsilon_{\text{mach}}/2$, então $1 + h = 1$ em ponto flutuante, a diferença progressiva dá derivada zero, e o erro é o próprio $\cos 1 \approx 0{,}54$.

### 5.3 Os bits de um número

O slider percorre números escolhidos para mostrar cada caso da seção 3.2: números exatos ($1$, $-2{,}5$), dízimas cortadas ($0{,}1$, $1/3$), o par $0{,}1 + 0{,}2$ e $0{,}3$, os extremos e os casos especiais. Compare as cores do expoente e da fração em cada um.

In [ ]:
exemplos = [("1", 1.0), ("−2,5", -2.5), ("0,1", 0.1), ("1/3", 1 / 3), ("0,1 + 0,2", 0.1 + 0.2), ("0,3", 0.3),
            ("maior", np.finfo(float).max), ("menor normal", np.finfo(float).tiny),
            ("menor subnormal", float(np.finfo(float).smallest_subnormal)), ("−0", -0.0),
            ("∞", float("inf")), ("NaN", float("nan"))]
quadros = [(nome, figura_bits(x, f"Os 64 bits de {nome}")) for nome, x in exemplos]
mostrar(com_controle(quadros, rotulo="número: ", ativo=4, titulos=[f"Os 64 bits de {n}" for n, _ in exemplos]))

**O que notar:** $0{,}1 + 0{,}2$ e $0{,}3$ diferem só no último bit da fração. O maior número tem todos os bits da fração iguais a 1, e o menor subnormal tem um único bit 1, o último. Em $\pm\infty$ e NaN, todos os bits do expoente são 1.

## 6. Trabalhando com o agente

Um contador de frequência mede um oscilador de quartzo de referência de 10 MHz. As leituras, em mHz (milésimos de hertz), ficam perto de $10^{10}$ mHz, com flutuações de alguns mHz. Queremos a **variância** das leituras, que mede a estabilidade do oscilador.

### 6.1 O pedido e o código

> *"Escreva uma função `variancia(dados)` que calcula a variância amostral de uma lista de números, numa única passagem pelos dados, para funcionar com sequências muito longas."*

O pedido parece razoável, mas o pedido de "uma única passagem" empurra o agente para a fórmula do livro de estatística, $\operatorname{Var}(x) = \overline{x^2} - \bar x^2$:

In [ ]:
# Código devolvido pelo agente (copiado sem alterações)
def variancia(dados):
    n = 0
    soma = 0.0
    soma_quadrados = 0.0
    for x in dados:
        n += 1
        soma += x
        soma_quadrados += x * x
    media = soma / n
    return (soma_quadrados - n * media**2) / (n - 1)

rng = np.random.default_rng(42)
ruido = rng.normal(0.0, 3.0, size=10_000)          # desvio padrão de 3 mHz
print("variância verdadeira ≈ 9;   código do agente com média 0:", numero(variancia(ruido), 4))

### 6.2 Verificação: um invariante

Somar uma constante a todos os dados **não muda** a variância. É um invariante matemático, e um ótimo teste. Deslocamos os dados, passo a passo, até a média real das leituras:

In [ ]:
deslocamentos = 10.0 ** np.arange(0, 11)
var_agente = [variancia(ruido + d) for d in deslocamentos]
var_numpy = [np.var(ruido + d, ddof=1) for d in deslocamentos]
for d, va in zip(deslocamentos[5:], var_agente[5:]):
    print(f"média {d:8.0e}:  agente = {va:12.4f}")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=deslocamentos, y=var_numpy, mode="lines+markers", name="np.var (duas passagens)",
                         line=dict(color=CORES["aproximacao"])))
fig.add_trace(go.Scatter(x=deslocamentos, y=var_agente, mode="lines+markers", name="código do agente",
                         line=dict(color=CORES["erro"])))
fig.update_xaxes(type="log", title="média dos dados (mHz)", exponentformat="power")
fig.update_yaxes(title="variância calculada (mHz²)", range=[-40, 60])
fig.update_layout(title="A variância deveria ser 9 para qualquer média")
mostrar(fig)

**O que notar:** até a média $10^6$ o erro do código do agente é pequeno, mas com $10^7$ ele já passa de 4 %. A partir de $10^8$ o código devolve uma **variância negativa**, o que é impossível. No caso do oscilador, com média $10^{10}$ mHz, o resultado é cerca de $-7\times 10^5$, sem nenhuma relação com o valor certo.

A causa é o cancelamento da seção 3.3. Os números $\overline{x^2}$ e $\bar x^2$ são da ordem de $10^{20}$ e concordam em quase todos os dígitos. A diferença entre eles, cerca de 9, se perde no arredondamento. O erro relativo cresce como $\varepsilon_{\text{mach}}\,(\bar x/\sigma)^2$: com $\bar x/\sigma \approx 3\times 10^9$, ele passa de $10^{3}$.

### 6.3 A correção

O **algoritmo de Welford** também faz uma passagem só, mas atualiza a média e a soma dos **desvios** a cada novo dado, sem nunca subtrair números grandes:

In [ ]:
def variancia_welford(dados):
    n, media, m2 = 0, 0.0, 0.0
    for x in dados:
        n += 1
        delta = x - media
        media += delta / n
        m2 += delta * (x - media)
    return m2 / (n - 1)

for d in [0.0, 1e6, 1e10]:
    compara(variancia_welford(ruido + d), np.var(ruido + d, ddof=1), rtol=1e-6)

> **Lição.** O código do agente está **matematicamente** correto. O erro é **numérico**, e só aparece com dados realistas. Testes com dados centrados em zero, que são os que um agente costuma escrever, nunca o encontrariam. Teste sempre com dados do tamanho e da escala do problema real, e use invariantes.

## 7. Quando falha

### 7.1 Somar muitos números pequenos

Ao somar números pequenos a um acumulador grande, os bits menos significativos se perdem, como no Patriot. Em precisão simples (`float32`, comum em GPUs e em aprendizado de máquina) isso aparece rápido. A **soma compensada de Kahan** guarda o erro perdido em cada soma e o devolve na próxima.

In [ ]:
def soma_ingenua(valores, tipo):
    s = tipo(0)
    for v in valores:
        s = tipo(s + v)
    return s

def soma_kahan(valores, tipo):
    s, comp = tipo(0), tipo(0)
    for v in valores:
        y = tipo(v - comp)
        t = tipo(s + y)
        comp = tipo((t - s) - y)
        s = t
    return s

valores = np.full(1_000_000, 0.1, dtype=np.float32)
print("valor exato:                 100000")
print("soma ingênua em float32:    ", soma_ingenua(valores, np.float32))
print("soma de Kahan em float32:   ", soma_kahan(valores, np.float32))

### 7.2 Overflow e underflow

O maior `float64` é cerca de $1{,}8\times10^{308}$. Acima disso, o resultado vira `inf`. Um caso frequente é a função **softmax** usada em aprendizado de máquina, $\sigma_i = e^{z_i} / \sum_j e^{z_j}$, que transborda para $z$ grande. Como a expressão não muda se subtrairmos a mesma constante de todos os $z_j$, a versão estável subtrai o máximo:

In [ ]:
z = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(all="ignore"):
    ingenua = np.exp(z) / np.sum(np.exp(z))
estavel = np.exp(z - z.max()) / np.sum(np.exp(z - z.max()))
print("softmax ingênua:", ingenua)
print("softmax estável:", estavel)

### 7.3 Comparar floats com `==`

Como vimos no capítulo 1, `0.1 + 0.2 == 0.3` é falso. Compare sempre com uma tolerância, de preferência relativa: `abs(a - b) <= rtol * max(abs(a), abs(b))`, ou use `math.isclose` e `np.isclose`.

### 7.4 Problema mal condicionado: nenhum algoritmo salva

A função $f(x) = 1 - \cos x$ perto de $x = 0$ sofre cancelamento, mas pode ser reescrita de forma estável como $2\sin^2(x/2)$. Já calcular $\tan x$ perto de $\pi/2$ é **mal condicionado**: $\kappa = |x / (\sin x \cos x)|$ cresce sem limite. Perto de $\pi/2$, mudar $x$ em um único bit, o menor passo possível em ponto flutuante, muda $\tan x$ já na sexta casa: o erro relativo da entrada, cerca de $10^{-16}$, é amplificado $\kappa \approx 10^{10}$ vezes. Nenhuma reescrita resolve, porque o problema é intrinsecamente sensível. O único remédio é saber $x$ com mais precisão.

In [ ]:
x = 1e-8
print(f"1 − cos(x)     = {1 - np.cos(x):.6e}   (cancelamento total)")
print(f"2 sen²(x/2)    = {2 * np.sin(x / 2)**2:.6e}   (estável; o exato é x²/2 = {x**2 / 2:.6e})")
x = np.pi / 2 - 1e-10
x_vizinho = np.nextafter(x, 2.0)            # o próximo float depois de x
print(f"tan(x)           = {np.tan(x):.10e}")
print(f"tan(x + 1 bit)   = {np.tan(x_vizinho):.10e}")
print(f"variação relativa: entrada {(x_vizinho - x) / x:.1e}, saída {(np.tan(x_vizinho) - np.tan(x)) / np.tan(x):.1e}")
print(f"κ = |x / (sen x cos x)| ≈ {abs(x / (np.sin(x) * np.cos(x))):.1e}")

## Resumo

| Fonte de erro | Origem | Como se comporta | Como combater |
| --- | --- | --- | --- |
| Arredondamento | número finito de bits | relativo, ≤ ε/2 ≈ 1,1×10⁻¹⁶ por operação | reescrever a expressão; somas compensadas; mais precisão |
| Cancelamento | subtração de números quase iguais | amplifica o erro relativo | reescrever (conjugado, identidades) |
| Truncamento | processo infinito tornado finito | $O(h^p)$, cai com $h$ | métodos de ordem mais alta |
| Mau condicionamento | o próprio problema | $\kappa \times$ erro da entrada | reformular o problema; não há algoritmo que resolva |
| Instabilidade | o algoritmo | erro muito maior que o inevitável | trocar de algoritmo |

## Exercícios

**Conceituais (sem computador)**

1. Escreva $0{,}2$ e $0{,}25$ em binário. Por que somar 0,25 repetidamente não acumula erro, mas somar 0,1 sim?
2. Calcule o número de condição de $f(x) = \sqrt{x}$, $f(x) = e^x$ e $f(x) = \ln x$. Qual delas é mal condicionada, e perto de que ponto?
3. Mostre que a fórmula alternativa de Bhaskara é algebricamente igual à original.
4. Estime, pela seção 3.5, o passo ótimo e o menor erro possível para a diferença **central**, que tem truncamento $\frac{|f'''|}{6}h^2$. Compare com o gráfico da seção 5.2.
5. Decodifique à mão o `float64` cujos bits, em hexadecimal, são `4059000000000000`. Depois codifique $-0{,}75$ e confira com o visualizador da seção 5.3.
6. Todo inteiro até $2^{53}$ é um `float64` exato. Explique por quê, usando a equação {eq}`eq-ieee754`, e encontre o menor inteiro positivo que **não** é um `float64`.

**Verificação (com agente)**

7. Peça a um agente uma função que calcule $e^x$ pela série de Taylor. Teste com $x = -30$. O resultado faz sentido? Explique o cancelamento e peça uma versão estável (dica: $e^{-30} = 1/e^{30}$).
8. Peça a um agente o cálculo do desvio padrão de uma série de dados de um sensor. Escreva o teste do invariante de deslocamento antes de ver o código, e use-o para avaliar a resposta.

**Projeto aberto**

9. **Precisão reduzida em aprendizado de máquina.** Treinamentos modernos usam `float16` e `bfloat16` para economizar memória. Compare a precisão da máquina e a faixa de valores desses formatos com `float32`. Repita a soma da seção 7.1 nesses formatos e explique por que bibliotecas como o PyTorch acumulam somas em precisão maior.

## Leituras

- D. Goldberg, "What every computer scientist should know about floating-point arithmetic", *ACM Computing Surveys* 23(1), 1991.
- N. J. Higham, *Accuracy and Stability of Numerical Algorithms*, 2ª ed., SIAM, 2002, capítulos 1 e 2.
- U.S. General Accounting Office, *Patriot Missile Defense: Software Problem Led to System Failure at Dhahran, Saudi Arabia*, GAO/IMTEC-92-26, 1992.
- S. C. Chapra e R. P. Canale, *Métodos Numéricos para Engenharia*, capítulos 3 e 4.